In [ ]:
% ============================================================
%
% This script is intended for system-level jitter budgeting.
%
% ============================================================
% MODELING ASSUMPTIONS
% ============================================================
%
% 1) All jitter quantities are RMS values.
% 2) Additive jitter contributions are assumed statistically
%    independent and are combined using root-sum-square (RSS).
% 3) Only random (RJ) components are modeled.
%    Deterministic jitter (DJ) is not included.
% 4) Supply-induced jitter, radiation effects, temperature drift,
%    aging, EMI, PCB coupling, and correlation effects are excluded.
% 5) This model evaluates clock-limited SNR only.
%    ADC quantization noise, thermal noise, and analog front-end
%    noise are not included.
%
% ============================================================

graphics_toolkit("qt")

clear all;
close all;
clc;

% ------------------------------------------------------------
% Automatic Report Output Directory
% ------------------------------------------------------------

scriptName = "SystemLevelJitterBudget";
reportRoot = fullfile("../Reports", scriptName);

if ~exist(reportRoot, "dir")
    mkdir(reportRoot);
end


f0 = 40e6;   % ADC sampling clock frequency (Hz)

% ------------------------------------------------------------
% 
% 1. Oscillator (Si510 LVDS)
% 
% Data from:
% https://eu.mouser.com/datasheet/3/564/1/si510_11.pdf
% Table 5. Output Clock Jitter and Phase Noise (LVDS)
% Row: Phase Jitter (RMS)
% 
% Note: we are using "12 kHz to 20 MHz integration bandwidth"
%       because it's the "industry standard". It's unrelated 
%       to ADC sampling frequency.
% 
% ------------------------------------------------------------

sigma_osc_typ = 0.8e-12;
sigma_osc_max = 1.0e-12;

% ------------------------------------------------------------
% 2. Fanout Buffer (Si53340, LVDS, VDD = 3.3 V)
% Data from:
% https://eu.mouser.com/datasheet/3/564/1/Si5334x_datasheet.pdf
% Table 3.6 – Additive Jitter (12 kHz – 20 MHz)
% 156.25 MHz row (closest to our operating region)
% ------------------------------------------------------------

sigma_fanout_typ = 150e-15; 
sigma_fanout_max = 200e-15;

% ------------------------------------------------------------
% 3. LVDS -> CMOS Buffer (PI6C49CB01Q)
% Data from:
% https://4donline.ihs.com/images/VipMasterIC/IC/DIOD/DIOD-S-A0009189280/DIOD-S-A0009189280-1.pdf?hkey=CECEF36DEECDED6468708AAF2E19C0C6
% Page 4, table with header "AC Characteristics, VDD = 3.3V ± 0.3V, TA = -40°C to 105°C"
% ------------------------------------------------------------

sigma_buffer = 0.15e-12;

% ------------------------------------------------------------
% 4. ADC Aperture Jitter (AD9238)
% https://www.analog.com/media/en/technical-documentation/data-sheets/ad9238.pdf
% Page 6 
% Row "Aperture Uncertainty"
% ------------------------------------------------------------

sigma_adc = 0.5e-12;

% ------------------------------------------------------------
% 5. TOTAL RMS JITTER AT ADC INPUT
% ------------------------------------------------------------

sigma_total_typ = sqrt( ...
    sigma_osc_typ^2 + ...
    sigma_fanout_typ^2 + ...
    sigma_buffer^2 + ...
    sigma_adc^2 );

sigma_total_max = sqrt( ...
    sigma_osc_max^2 + ...
    sigma_fanout_max^2 + ...
    sigma_buffer^2 + ...
    sigma_adc^2 );

fprintf("\n--- Jitter Contributions (Typical Oscillator) ---\n");
fprintf("Oscillator:      %.3f ps\n", sigma_osc_typ*1e12);
fprintf("Fanout:          %.3f ps\n", sigma_fanout_typ*1e12);
fprintf("LVDS->CMOS:       %.3f ps\n", sigma_buffer*1e12);
fprintf("ADC aperture:    %.3f ps\n", sigma_adc*1e12);
fprintf("Total RMS:       %.3f ps\n", sigma_total_typ*1e12);

fprintf("\n--- Jitter Contributions (Maximum Oscillator) ---\n");
fprintf("Oscillator:      %.3f ps\n", sigma_osc_max*1e12);
fprintf("Fanout:          %.3f ps\n", sigma_fanout_max*1e12);
fprintf("Total RMS:       %.3f ps\n", sigma_total_max*1e12);

% ------------------------------------------------------------
% PERFORMANCE AT CRITICAL OPERATING POINTS
% ------------------------------------------------------------

fs_1 = 40e6;
fs_2 = 65e6;

f_nyq_1 = fs_1 / 2;   % 20 MHz
f_nyq_2 = fs_2 / 2;   % 32.5 MHz

% --- Typical ---
SNR_40_typ = -20 * log10(2*pi*f_nyq_1*sigma_total_typ);
ENOB_40_typ = (SNR_40_typ - 1.76)/6.02;

SNR_65_typ = -20 * log10(2*pi*f_nyq_2*sigma_total_typ);
ENOB_65_typ = (SNR_65_typ - 1.76)/6.02;

% --- Maximum ---
SNR_40_max = -20 * log10(2*pi*f_nyq_1*sigma_total_max);
ENOB_40_max = (SNR_40_max - 1.76)/6.02;

SNR_65_max = -20 * log10(2*pi*f_nyq_2*sigma_total_max);
ENOB_65_max = (SNR_65_max - 1.76)/6.02;

fprintf("\n--- Jitter-Limited Performance at Nyquist (Typical) ---\n");
fprintf("40 MSPS:  SNR = %.2f dB | ENOB = %.2f bits\n", SNR_40_typ, ENOB_40_typ);
fprintf("65 MSPS:  SNR = %.2f dB | ENOB = %.2f bits\n", SNR_65_typ, ENOB_65_typ);

fprintf("\n--- Jitter-Limited Performance at Nyquist (Maximum) ---\n");
fprintf("40 MSPS:  SNR = %.2f dB | ENOB = %.2f bits\n", SNR_40_max, ENOB_40_max);
fprintf("65 MSPS:  SNR = %.2f dB | ENOB = %.2f bits\n", SNR_65_max, ENOB_65_max);

% ------------------------------------------------------------
% 6. JITTER-LIMITED SNR AND ENOB MODEL
% ------------------------------------------------------------

f_in = logspace(3, log10(f_nyq_2), 2000);

SNR_jitter_typ = -20 .* log10(2*pi .* f_in .* sigma_total_typ);
ENOB_jitter_typ = (SNR_jitter_typ - 1.76) ./ 6.02;

SNR_jitter_max = -20 .* log10(2*pi .* f_in .* sigma_total_max);
ENOB_jitter_max = (SNR_jitter_max - 1.76) ./ 6.02;


% ------------------------------------------------------------
% High-Resolution SNR Plot (SVG Export)
% ------------------------------------------------------------

figure("Position", [100 100 1200 800]);

semilogx(f_in, SNR_jitter_typ, "LineWidth", 2);
hold on;
semilogx(f_in, SNR_jitter_max, "--", "LineWidth", 2);

grid on;
set(gca, "FontSize", 14);

xlabel("Input Frequency (Hz)", "FontSize", 16);
ylabel("Jitter-Limited SNR (dB)", "FontSize", 16);
title("System-Level Clock-Limited SNR vs Input Frequency", "FontSize", 18);

legend("Typical", "Maximum", "Location", "southwest");

set(gcf, "PaperPositionMode", "auto");
print(gcf, fullfile(reportRoot, "SNRJitter.svg"), "-dsvg");

% ------------------------------------------------------------
% High-Resolution ENOB Plot (SVG Export)
% ------------------------------------------------------------

figure("Position", [100 100 1200 800]);

semilogx(f_in, ENOB_jitter_typ, "LineWidth", 2);
hold on;
semilogx(f_in, ENOB_jitter_max, "--", "LineWidth", 2);

grid on;
set(gca, "FontSize", 14);

xlabel("Input Frequency (Hz)", "FontSize", 16);
ylabel("Effective Number of Bits (ENOB)", "FontSize", 16);
title("System-Level Clock-Limited ENOB vs Input Frequency", "FontSize", 18);

legend("Typical", "Maximum", "Location", "southwest");

set(gcf, "PaperPositionMode", "auto");
print(gcf, fullfile(reportRoot, "ENOBJitter.svg"), "-dsvg");

% ------------------------------------------------------------
% 7. JITTER CONTRIBUTION PIE CHART (Typical Case)
% ------------------------------------------------------------
%
% Jitter adds in variance (sigma^2).
% Contribution percentage is computed from
% variance ratio, not sigma ratio.
%

% Variance contributions (typical case)
var_osc     = sigma_osc_typ^2;
var_fanout  = sigma_fanout_typ^2;
var_buffer  = sigma_buffer^2;
var_adc     = sigma_adc^2;

var_total = sigma_total_typ^2;

% Percentage contribution
pct = 100 * [ ...
    var_osc, ...
    var_fanout, ...
    var_buffer, ...
    var_adc ] / var_total;

figure("Position", [100 100 1200 800]);
pie(pct);
title("Jitter Variance Contribution (Typical Case)", "FontSize", 16);
legend( ...
    sprintf("Oscillator (%.1f%%)", pct(1)), ...
    sprintf("Fanout (%.1f%%)", pct(2)), ...
    sprintf("LVDS->CMOS (%.1f%%)", pct(3)), ...
    sprintf("ADC Aperture (%.1f%%)", pct(4)), ...
    "Location", "eastoutside");

set(gca, "FontSize", 14);

set(gcf, "PaperPositionMode", "auto");
print(gcf, fullfile(reportRoot, "jitterPieTypical.svg"), "-dsvg");




--- Jitter Contributions (Typical Oscillator) ---
Oscillator:      0.800 ps
Fanout:          0.150 ps
LVDS->CMOS:       0.150 ps
ADC aperture:    0.500 ps
Total RMS:       0.967 ps

--- Jitter Contributions (Maximum Oscillator) ---
Oscillator:      1.000 ps
Fanout:          0.200 ps
Total RMS:       1.146 ps

--- Jitter-Limited Performance at Nyquist (Typical) ---
40 MSPS:  SNR = 78.31 dB | ENOB = 12.72 bits
65 MSPS:  SNR = 74.09 dB | ENOB = 12.02 bits

--- Jitter-Limited Performance at Nyquist (Maximum) ---
40 MSPS:  SNR = 76.83 dB | ENOB = 12.47 bits
65 MSPS:  SNR = 72.62 dB | ENOB = 11.77 bits
QStandardPaths: XDG_RUNTIME_DIR not set, defaulting to '/tmp/runtime-root'
QStandardPaths: XDG_RUNTIME_DIR not set, defaulting to '/tmp/runtime-root'
